# 🔥 Firecrawl Studio — Enhanced Notebook

### Features:
- ⚡ **Multi-URL Scraper** — Extract from up to 5 URLs simultaneously
- 🧬 **Advanced Schema Builder** — Typed fields, descriptions, required flags, list types
- 🕑 **Query History** — Track and reuse past extractions
- 📊 **Live Preview** — See extracted data as a table

---

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -U firecrawl-py python-dotenv pydantic pandas tabulate

## 🔑 Step 2 — Setup API Key

In [ ]:
import os
from dotenv import load_dotenv
from firecrawl import FirecrawlApp
from pydantic import BaseModel
import pandas as pd
from typing import Any, Dict, List, Optional
import json
from datetime import datetime

# Load from .env file OR set directly below
load_dotenv()
FIRECRAWL_API_KEY = os.getenv("FIRECRAWL_API_KEY") or "YOUR_API_KEY_HERE"

app = FirecrawlApp(api_key=FIRECRAWL_API_KEY)
print("✅ Firecrawl client ready!")

---
## ⚡ FEATURE 1 — Multi-URL Scraper

Extract structured data from **multiple URLs at once** using a single natural language prompt.
Supports up to 5 URLs per run with per-source result display and query history.

In [ ]:
# ── Multi-URL Scraper Class ──────────────────────────────────────────────────

class MultiURLScraper:
    MAX_URLS = 5

    def __init__(self, firecrawl_app: FirecrawlApp):
        self.app = firecrawl_app
        self.history: List[Dict] = []  # query history

    def validate_urls(self, urls: List[str]) -> List[str]:
        """Filter out empty/invalid URLs and enforce MAX_URLS limit."""
        valid = [u.strip() for u in urls if u.strip()]
        if not valid:
            raise ValueError("Provide at least one valid URL.")
        if len(valid) > self.MAX_URLS:
            print(f"⚠️  Only first {self.MAX_URLS} URLs will be used.")
            valid = valid[:self.MAX_URLS]
        return valid

    def extract(self, urls: List[str], prompt: str) -> Dict[str, Any]:
        """
        Run extraction across multiple URLs.
        Returns a dict: {url -> extracted_data}
        """
        valid_urls = self.validate_urls(urls)
        print(f"🌐 Extracting from {len(valid_urls)} URL(s)...")
        print(f"💬 Prompt: {prompt}\n")

        results = {}
        for url in valid_urls:
            try:
                print(f"  → {url}")
                raw = self.app.extract([url], {"prompt": prompt})
                results[url] = raw.get("data", raw)
            except Exception as e:
                results[url] = {"error": str(e)}

        # Save to history
        self.history.append({
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "urls": valid_urls,
            "prompt": prompt,
            "result_count": len(results),
        })

        return results

    def display_results(self, results: Dict[str, Any]):
        """Pretty-print results per source URL."""
        for url, data in results.items():
            print(f"\n{'─'*60}")
            print(f"🔗 Source: {url}")
            print(f"{'─'*60}")
            if "error" in data:
                print(f"  ❌ Error: {data['error']}")
            elif isinstance(data, list):
                df = pd.DataFrame(data)
                print(df.to_markdown(index=False))
            elif isinstance(data, dict):
                df = pd.DataFrame([data])
                print(df.to_markdown(index=False))
            else:
                print(data)

    def show_history(self):
        """Display query history."""
        if not self.history:
            print("No history yet.")
            return
        print("\n🕑 Query History")
        print("─" * 60)
        for i, h in enumerate(reversed(self.history), 1):
            print(f"[{i}] {h['timestamp']} | {len(h['urls'])} URL(s) | Prompt: {h['prompt'][:60]}")

print("✅ MultiURLScraper class ready!")

In [ ]:
# ── Run Multi-URL Extraction ─────────────────────────────────────────────────

scraper = MultiURLScraper(app)

# 👇 Edit these URLs and prompt
urls = [
    "https://blog.dailydoseofds.com/*",
    "https://towardsdatascience.com",
    # Add up to 5 URLs
]

prompt = "Extract article title, publish date, and article link for all articles related to LLMs"

results = scraper.extract(urls, prompt)
scraper.display_results(results)

In [ ]:
# ── View Query History ───────────────────────────────────────────────────────
scraper.show_history()

In [ ]:
# ── Reuse a Past Query ───────────────────────────────────────────────────────
# Pick an index from history (0 = most recent)

past = scraper.history[-1]  # get most recent
print(f"Reusing: {past['prompt']}")
print(f"URLs: {past['urls']}")

# Uncomment to re-run:
# results2 = scraper.extract(past["urls"], past["prompt"])
# scraper.display_results(results2)

---
## 🧬 FEATURE 2 — Advanced Schema Builder

Define **typed, described, validated** schemas for structured extraction.
Supports: `str`, `bool`, `int`, `float`, `list[str]`, `list[int]` — with required flags and field descriptions.

In [ ]:
# ── Advanced Schema Builder Class ────────────────────────────────────────────

SUPPORTED_TYPES = ["str", "bool", "int", "float", "list[str]", "list[int]"]

class AdvancedSchemaBuilder:
    MAX_FIELDS = 10

    def __init__(self, name: str = "MySchema"):
        self.name = name
        self.fields: List[Dict] = []

    def add_field(
        self,
        name: str,
        field_type: str = "str",
        description: str = "",
        required: bool = True
    ) -> "AdvancedSchemaBuilder":
        """Add a field. Returns self for chaining."""
        if len(self.fields) >= self.MAX_FIELDS:
            raise ValueError(f"Max {self.MAX_FIELDS} fields allowed.")
        if field_type not in SUPPORTED_TYPES:
            raise ValueError(f"Type '{field_type}' not supported. Use: {SUPPORTED_TYPES}")
        self.fields.append({
            "name": name,
            "type": field_type,
            "description": description,
            "required": required,
        })
        return self  # enable chaining

    def remove_field(self, name: str) -> "AdvancedSchemaBuilder":
        self.fields = [f for f in self.fields if f["name"] != name]
        return self

    def _type_to_json(self, t: str) -> dict:
        """Convert string type to JSON Schema type definition."""
        mapping = {
            "str":       {"type": "string"},
            "bool":      {"type": "boolean"},
            "int":       {"type": "integer"},
            "float":     {"type": "number"},
            "list[str]": {"type": "array", "items": {"type": "string"}},
            "list[int]": {"type": "array", "items": {"type": "integer"}},
        }
        return mapping[t]

    def build(self) -> dict:
        """Build and return the JSON Schema dict."""
        if not self.fields:
            raise ValueError("Add at least one field before building.")
        properties = {}
        required = []
        for f in self.fields:
            prop = self._type_to_json(f["type"])
            if f["description"]:
                prop["description"] = f["description"]
            properties[f["name"]] = prop
            if f["required"]:
                required.append(f["name"])
        return {
            "title": self.name,
            "type": "object",
            "properties": properties,
            "required": required,
        }

    def preview(self):
        """Print a human-readable summary of the schema."""
        print(f"\n📋 Schema: {self.name}")
        print("─" * 60)
        rows = []
        for f in self.fields:
            rows.append({
                "Field": f["name"],
                "Type": f["type"],
                "Required": "✅" if f["required"] else "—",
                "Description": f["description"] or "(none)",
            })
        print(pd.DataFrame(rows).to_markdown(index=False))

    def print_json(self):
        """Print the raw JSON Schema."""
        print(json.dumps(self.build(), indent=2))

print("✅ AdvancedSchemaBuilder class ready!")

In [ ]:
# ── Build a Schema (chained) ─────────────────────────────────────────────────

schema_builder = (
    AdvancedSchemaBuilder(name="ArticleSchema")
    .add_field("article_title",  "str",       "Title of the article",             required=True)
    .add_field("publish_date",   "str",       "Date the article was published",    required=True)
    .add_field("article_link",   "str",       "Full URL link to the article",      required=True)
    .add_field("tags",           "list[str]", "Topic tags associated",             required=False)
    .add_field("read_time_mins", "int",       "Estimated read time in minutes",    required=False)
    .add_field("is_premium",     "bool",      "Whether it is behind a paywall",    required=False)
)

# Preview the schema
schema_builder.preview()

In [ ]:
# ── Print Raw JSON Schema ────────────────────────────────────────────────────
schema_builder.print_json()

In [ ]:
# ── Run Extraction with Advanced Schema ─────────────────────────────────────

target_url = "https://blog.dailydoseofds.com/*"  # 👈 change this
prompt_text = "Extract all articles related to LLMs with their title, date, link, tags and read time"

schema = schema_builder.build()

data = app.extract(
    [target_url],
    {
        "prompt": prompt_text,
        "schema": schema,
    }
)

print("✅ Raw Response:")
print(json.dumps(data, indent=2))

In [ ]:
# ── Display as Table ─────────────────────────────────────────────────────────

extracted = data.get("data", data)

if isinstance(extracted, list):
    df = pd.DataFrame(extracted)
elif isinstance(extracted, dict):
    # Check if there's a nested list under the first key
    first_key = list(extracted.keys())[0]
    if isinstance(extracted[first_key], list):
        df = pd.DataFrame(extracted[first_key])
    else:
        df = pd.DataFrame([extracted])

print(f"📊 Extracted {len(df)} record(s)\n")
print(df.to_markdown(index=False))

---
## 🔁 Combine Both Features

Use **Multi-URL Scraper** + **Advanced Schema** together for maximum power.

In [ ]:
# ── Combined: Multi-URL + Schema ─────────────────────────────────────────────

class FirecrawlStudio:
    """Combines MultiURLScraper + AdvancedSchemaBuilder into one interface."""

    def __init__(self, firecrawl_app: FirecrawlApp):
        self.app = firecrawl_app
        self.history: List[Dict] = []

    def extract(
        self,
        urls: List[str],
        prompt: str,
        schema_builder: Optional[AdvancedSchemaBuilder] = None
    ) -> Dict[str, Any]:
        valid_urls = [u.strip() for u in urls if u.strip()][:5]
        extract_params = {"prompt": prompt}
        if schema_builder:
            extract_params["schema"] = schema_builder.build()
            print(f"🧬 Using schema: {schema_builder.name} ({len(schema_builder.fields)} fields)")

        results = {}
        for url in valid_urls:
            try:
                raw = self.app.extract([url], extract_params)
                results[url] = raw.get("data", raw)
            except Exception as e:
                results[url] = {"error": str(e)}

        self.history.append({
            "timestamp": datetime.now().strftime("%H:%M:%S"),
            "urls": valid_urls,
            "prompt": prompt,
            "schema": schema_builder.name if schema_builder else None,
        })
        return results

    def display(self, results: Dict[str, Any]):
        for url, data in results.items():
            print(f"\n{'═'*60}")
            print(f"🔗 {url}")
            print(f"{'─'*60}")
            if "error" in data:
                print(f"❌ {data['error']}")
            elif isinstance(data, list):
                print(pd.DataFrame(data).to_markdown(index=False))
            elif isinstance(data, dict):
                print(pd.DataFrame([data]).to_markdown(index=False))

    def show_history(self):
        print("\n🕑 Session History")
        print("─" * 60)
        for i, h in enumerate(reversed(self.history), 1):
            schema_info = f" | Schema: {h['schema']}" if h['schema'] else ""
            print(f"[{i}] {h['timestamp']} | {len(h['urls'])} URL(s){schema_info}")
            print(f"     Prompt: {h['prompt'][:70]}")


# ── Example Usage ─────────────────────────────────────────────────────────────
studio = FirecrawlStudio(app)

# Build schema
product_schema = (
    AdvancedSchemaBuilder("ProductSchema")
    .add_field("product_name",  "str",       "Name of the product",         required=True)
    .add_field("price",         "float",     "Price in USD",                required=True)
    .add_field("in_stock",      "bool",      "Whether item is in stock",    required=True)
    .add_field("features",      "list[str]", "Key product features",        required=False)
    .add_field("rating",        "float",     "Average customer rating",     required=False)
)

product_schema.preview()

# Run (replace URLs with real ones)
# results = studio.extract(
#     urls=["https://store1.com/products", "https://store2.com/shop"],
#     prompt="Extract all product listings with name, price, stock status, features and rating",
#     schema_builder=product_schema
# )
# studio.display(results)
# studio.show_history()

print("\n✅ FirecrawlStudio ready! Uncomment the run block above to execute.")

---
## 📝 Quick Reference

| Feature | Class | Key Method |
|---|---|---|
| Multi-URL Scraper | `MultiURLScraper` | `.extract(urls, prompt)` |
| Schema Builder | `AdvancedSchemaBuilder` | `.add_field(...).build()` |
| Combined | `FirecrawlStudio` | `.extract(urls, prompt, schema_builder)` |

**Supported field types:** `str` · `bool` · `int` · `float` · `list[str]` · `list[int]`

**Limits:** Max 5 URLs per extraction · Max 10 fields per schema